### Project DRISHTI: Lunar South Pole Navigation & Hazard Pipeline
**Objective:** Map traversable terrain, identify Permanently Shadowed Regions (PSRs), and compute an optimal rover path to subsurface water-ice inside the Shoemaker Crater region using DFSAR, LOLA, and LRO illumination data.

---

#### Phase 0 & 1: Environment Initialization and Spatial Overlap Audit
**Scientific Rationale:** Before applying any gradient mathematics or morphological filters, we must mathematically guarantee that our Topography (DEM) and Illumination datasets geographically encompass our target Radar Swath (CPR/Ice Map). 

Warping a small regional tile into a large radar bounding box causes GDAL to pad the void space with `0.0` values. In planetary GIS, algorithms interpret `0.0` elevation or slope as perfectly safe, flat terrain, leading to catastrophic false-positive landing site selections (the "Ghost Pixel" problem). 

This cell initializes our strict `-9999.0` NoData environment and audits the bounding boxes of the raw PDS/PGDA files against the DFSAR radar swath to confirm 100% true spatial overlap.

In [1]:
import rasterio
import numpy as np
from scipy.ndimage import (distance_transform_edt, binary_dilation, 
                           label, maximum_filter, minimum_filter,
                           generic_filter)
import os

# --- GLOBAL CONSTANTS ---
NODATA = -9999.0
BASE_DIR = r"C:\DRISHTI_POC"


def spatial_overlap_audit_v2():
    print("=" * 70)
    print("PHASE 1: SPATIAL OVERLAP AUDIT (Raw Source vs Radar Swath)")
    print("=" * 70)
    
    # Files we are checking
    files = {
        "DFSAR_RADAR_SWATH": os.path.join(BASE_DIR, "WARPED_CPR.tif"),
        "ICE_TARGETS": os.path.join(BASE_DIR, "DRISHTI_FINAL_ICE_MAP.tif"),
        "NEW_LOLA_DEM (LBL)": os.path.join(BASE_DIR, "LDEM_85S_10M.LBL"),
        "NEW_AVG_ILLUM": os.path.join(BASE_DIR, "AVGVISIB_85S_060M_201608.tiff"),
        "NEW_PSR_MASK": os.path.join(BASE_DIR, "LPSR_85S_060M_201608.tiff")
    }
    
    radar_bounds = None
    
    for name, fpath in files.items():
        if not os.path.exists(fpath):
            print(f"\n[{name}] FILE NOT FOUND: Check path -> {fpath}")
            continue
            
        try:
            with rasterio.open(fpath) as src:
                b = src.bounds
                crs = src.crs
                
                print(f"\n{name}")
                print(f"  Bounds: Left={b.left:.1f}, Right={b.right:.1f}, Bottom={b.bottom:.1f}, Top={b.top:.1f}")
                print(f"  Size:   {src.width} x {src.height} pixels")
                print(f"  CRS:    {crs.data['proj'] if crs and 'proj' in crs.data else crs}")
                
                if name == "DFSAR_RADAR_SWATH":
                    radar_bounds = b
                elif radar_bounds:
                    # Check if the new dataset completely encompasses the radar swath
                    covers_x = (b.left <= radar_bounds.left) and (b.right >= radar_bounds.right)
                    covers_y = (b.bottom <= radar_bounds.bottom) and (b.top >= radar_bounds.top)
                    
                    if covers_x and covers_y:
                        print("  ✅ STATUS: Completely encompasses the DFSAR Radar Swath. Safe for warping.")
                    else:
                        print("  ❌ STATUS: Does NOT fully encompass the Radar Swath. Data gaps will occur.")
        except Exception as e:
            print(f"\n[{name}] ERROR reading file: {e}")

# Execute Phase 1
spatial_overlap_audit_v2()

PHASE 1: SPATIAL OVERLAP AUDIT (Raw Source vs Radar Swath)

DFSAR_RADAR_SWATH
  Bounds: Left=-27473.8, Right=127476.2, Bottom=12328.7, Top=90053.7
  Size:   6198 x 3109 pixels
  CRS:    stere

ICE_TARGETS
  Bounds: Left=-27473.8, Right=127476.2, Bottom=12328.7, Top=90053.7
  Size:   6198 x 3109 pixels
  CRS:    stere
  ✅ STATUS: Completely encompasses the DFSAR Radar Swath. Safe for warping.

NEW_LOLA_DEM (LBL)
  Bounds: Left=-151680.0, Right=151680.0, Bottom=-151680.0, Top=151680.0
  Size:   30336 x 30336 pixels
  CRS:    stere
  ✅ STATUS: Completely encompasses the DFSAR Radar Swath. Safe for warping.

NEW_AVG_ILLUM
  Bounds: Left=-151740.0, Right=151740.0, Bottom=-151740.0, Top=151740.0
  Size:   5058 x 5058 pixels
  CRS:    stere
  ✅ STATUS: Completely encompasses the DFSAR Radar Swath. Safe for warping.

NEW_PSR_MASK
  Bounds: Left=-151740.0, Right=151740.0, Bottom=-151740.0, Top=151740.0
  Size:   5058 x 5058 pixels
  CRS:    stere
  ✅ STATUS: Completely encompasses the DFSAR Rad

### Phase 2: Safe Ingestion, Scaling, and Master Grid Warping

### 1. Scientific Rationale & Core Physics
Raw planetary data arrays are distributed in specialized binary formats. For the Lunar Orbiter Laser Altimeter (LOLA), data is stored as a detached ASCII text label (`.LBL`) which maps to an uncompressed 16-bit binary image (`.IMG`). 
Attempting to parse a raw `.IMG` file directly via standard image processors causes an immediate I/O error because it lacks spatial coordinate headers. Furthermore, the values stored inside are Digital Numbers (DN) rather than metrics in meters. According to the PDS3 label metadata:
$$\text{Elevation (meters)} = \text{DN} \times 0.5$$
If this scaling property is not applied directly upon array ingestion, the relative topography flatlines by exactly 50%, resulting in invalid derivative gradients during slope calculation.

##### 2. Algorithmic Strategy

We isolate the coordinate reference system (CRS) and precise affine transform matrix from our master radar swath (`WARPED_CPR.tif`). The raw binary numbers are extracted by opening the `.LBL` file (letting GDAL parse the binary pointer natively), multiplied by the `0.5` physical scaling factor, and mathematically projected onto the master 25-meter grid. Out-of-bounds space is strictly encoded to a floating-point `NODATA = -9999.0`. 

For the solar illumination data (`AVGVISIB`) and the permanent shadow mask (`LPSR`), which have a native spacing of 60 meters per pixel, we warp them into the same grid using `Resampling.bilinear` and `Resampling.nearest` respectively to ensure crisp spatial coregistration without blurring categorical masks.

In [2]:
import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np
import os

# Define absolute filepaths based on verified audit locations
LDEM_LBL_PATH = os.path.join(BASE_DIR, "LDEM_85S_10M.LBL")
AVGVISIB_PATH = os.path.join(BASE_DIR, "AVGVISIB_85S_060M_201608.tiff")
LPSR_PATH     = os.path.join(BASE_DIR, "LPSR_85S_060M_201608.tiff")
REF_GRID_PATH = os.path.join(BASE_DIR, "WARPED_CPR.tif")

# Target Master Phase 2 Output GeoTIFFs
OUT_WARPED_DEM   = os.path.join(BASE_DIR, "LDEM85S_WARPED_25m.tif")
OUT_WARPED_ILLUM = os.path.join(BASE_DIR, "AVGVISIB_WARPED_25m.tif")
OUT_WARPED_PSR   = os.path.join(BASE_DIR, "PSR_MASK_25m.tif")

def execute_phase2_ingest():
    print("=" * 70)
    print("PHASE 2: EXECUTING NATIVE PDS/PGDA INGESTION & GRID ALIGNMENT")
    print("=" * 70)
    
    # Extract authoritative spatial framework from Phase 1 radar baseline
    with rasterio.open(REF_GRID_PATH) as ref:
        dst_transform = ref.transform
        dst_crs       = ref.crs
        dst_shape     = (ref.height, ref.width)
        master_profile = ref.profile.copy()
    
    print(f"Target Sync Framework: {dst_shape[1]}x{dst_shape[0]} matrix at 25.0m resolution.")
    
    # ----------------------------------------------------------------
    # LAYER 1: NATIVE PDS3 ELEVATION MODEL (LDEM)
    # ----------------------------------------------------------------
    print("\nProcessing Topographic Ingestion layer...")
    # Open the .LBL file. Rasterio/GDAL uses the label metadata to parse the .IMG binary stream.
    with rasterio.open(LDEM_LBL_PATH) as src:
        # Read the raw binary table into memory as float32 to hold decimal precision after scaling
        raw_dn = src.read(1).astype(np.float32)
        
        # Enforce the explicit scaling multiplier dictated by NASA metadata
        print("Applying planetary scaling modifier: DN * 0.5 meters...")
        scaled_dem = raw_dn * 0.5
        
        # Allocate clean destination canvas pre-padded with un-traversable NoData bounds
        dem_dst_array = np.full(dst_shape, NODATA, dtype=np.float32)
        
        print("Executing bilinear surface grid projection...")
        reproject(
            source=scaled_dem,
            destination=dem_dst_array,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            resampling=Resampling.bilinear
        )
        
    # Write the clean, uncorrupted elevation master raster
    master_profile.update(dtype=rasterio.float32, count=1, compress='lzw', nodata=NODATA)
    with rasterio.open(OUT_WARPED_DEM, 'w', **master_profile) as dst:
        dst.write(dem_dst_array, 1)
    print(f"[SUCCESS] Elevation Layer written: {OUT_WARPED_DEM}")
    
    # ----------------------------------------------------------------
    # LAYER 2: TIME-INTEGRATED ANNUAL ILLUMINATION (AVGVISIB)
    # ----------------------------------------------------------------
    print("\nProcessing Solar Illumination layer...")
    with rasterio.open(AVGVISIB_PATH) as src:
        raw_illum = src.read(1).astype(np.float32)
        
        # Clean any default system nodata boundaries from PGDA
        if src.nodata is not None:
            raw_illum[raw_illum == src.nodata] = np.nan
            
        illum_dst_array = np.full(dst_shape, NODATA, dtype=np.float32)
        
        print("Executing continuous bilinear downsampling from 60m to 25m...")
        reproject(
            source=raw_illum,
            destination=illum_dst_array,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            resampling=Resampling.bilinear
        )
        
    with rasterio.open(OUT_WARPED_ILLUM, 'w', **master_profile) as dst:
        dst.write(illum_dst_array, 1)
    print(f"[SUCCESS] Solar Illumination written: {OUT_WARPED_ILLUM}")

    # ----------------------------------------------------------------
    # LAYER 3: BINARY CUMULATIVE PSR MASK (LPSR)
    # ----------------------------------------------------------------
    print("\nProcessing Permanently Shadowed Region mask...")
    with rasterio.open(LPSR_PATH) as src:
        raw_psr = src.read(1).astype(np.float32)
        
        psr_dst_array = np.full(dst_shape, NODATA, dtype=np.float32)
        
        # CRITICAL MARGIN: We use nearest-neighbor for a binary mask to protect
        # the sharp 0/1 boolean structure. Bilinear would introduce fractional data blur.
        print("Executing categorical nearest-neighbor allocation...")
        reproject(
            source=raw_psr,
            destination=psr_dst_array,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            resampling=Resampling.nearest
        )
        
    with rasterio.open(OUT_WARPED_PSR, 'w', **master_profile) as dst:
        dst.write(psr_dst_array, 1)
    print(f"[SUCCESS] PSR Binary Mask written: {OUT_WARPED_PSR}")

# Execute Phase 2 Ingestion Block
execute_phase2_ingest()

PHASE 2: EXECUTING NATIVE PDS/PGDA INGESTION & GRID ALIGNMENT
Target Sync Framework: 6198x3109 matrix at 25.0m resolution.

Processing Topographic Ingestion layer...
Applying planetary scaling modifier: DN * 0.5 meters...
Executing bilinear surface grid projection...
[SUCCESS] Elevation Layer written: C:\DRISHTI_POC\LDEM85S_WARPED_25m.tif

Processing Solar Illumination layer...
Executing continuous bilinear downsampling from 60m to 25m...
[SUCCESS] Solar Illumination written: C:\DRISHTI_POC\AVGVISIB_WARPED_25m.tif

Processing Permanently Shadowed Region mask...
Executing categorical nearest-neighbor allocation...
[SUCCESS] PSR Binary Mask written: C:\DRISHTI_POC\PSR_MASK_25m.tif


### Phase 3: Terrain Hazard Derivation (Slope & Roughness)

##### 1. Scientific Rationale
Raw elevation data tells us *where* the rover is, but not *if* the rover can survive there. We must translate absolute elevation ($Z$) into two strict mechanical hazard constraints for the Chandrayaan-3/Pragyan chassis:
1. **Slope (Rover Tip-Over Risk):** Calculated in degrees. The operational limit is $\le 10^\circ$.
2. **Roughness (Suspension/Boulder Hazard):** Calculated as the standard deviation of elevation ($\sigma_z$) within a localized moving window. 

##### 2. Algorithmic Strategy
* **Slope:** We compute the partial derivatives (gradients) of the elevation matrix in the X and Y directions using central differences. The absolute slope angle $\theta$ is derived via $\theta = \arctan(\sqrt{(\frac{\partial z}{\partial x})^2 + (\frac{\partial z}{\partial y})^2})$. 
* **Roughness:** We pass a 3x3 pixel moving window (representing a 75m x 75m physical patch of lunar regolith) over the DEM. We calculate the `np.nanstd` (Standard Deviation, ignoring the NoData vacuum). A high standard deviation means the 75m patch contains severe elevation spikes (boulders or sharp craters).

*Note on Computation:* Because we are processing a massive 19-million pixel grid and safely navigating `NaN` boundaries, the roughness standard deviation filter may take 1 to 3 minutes to compute. This is standard for high-fidelity planetary GIS pipelines.

In [3]:
import rasterio
import numpy as np
from scipy.ndimage import generic_filter
import os

# --- PATHS ---
BASE_DIR = r"C:\DRISHTI_POC"
IN_DEM = os.path.join(BASE_DIR, "LDEM85S_WARPED_25m.tif")

OUT_SLOPE = os.path.join(BASE_DIR, "FINAL_DRISHTI_Slope_25m.tif")
OUT_ROUGH = os.path.join(BASE_DIR, "FINAL_DRISHTI_Roughness_25m.tif")
NODATA = -9999.0

def compute_terrain_hazards():
    print("=" * 70)
    print("PHASE 3: DERIVING SLOPE AND ROUGHNESS KINEMATICS")
    print("=" * 70)
    
    with rasterio.open(IN_DEM) as src:
        dem = src.read(1).astype(np.float32)
        meta = src.profile.copy()
        pixel_size_m = abs(src.transform.a) # Should be 25.0m
        
    print(f"Loaded Master DEM. Pixel Resolution: {pixel_size_m}m")
    
    # 1. MASK OUT THE VACUUM
    # Convert NODATA boundaries to np.nan so they don't create 9,000-meter fake cliffs
    valid_mask = (dem != NODATA)
    dem_nan = np.where(valid_mask, dem, np.nan)
    
    # 2. CALCULATE SLOPE
    print("Calculating spatial gradients (Slope)...")
    # np.gradient computes central differences. We divide by pixel size to get dz/dx in meters.
    dy, dx = np.gradient(dem_nan, pixel_size_m, pixel_size_m)
    
    # Calculate hypotenuse of gradients, then arctan for the angle, then convert to degrees
    slope_rad = np.arctan(np.hypot(dx, dy))
    slope_deg = np.degrees(slope_rad)
    
    # 3. CALCULATE ROUGHNESS (Standard Deviation Proxy)
    print("Calculating surface roughness (3x3 localized standard deviation)...")
    print("  -> (This may take 1-3 minutes on a 19-million pixel grid. Please wait...)")
    
    # generic_filter applies the function (np.nanstd) to a rolling 3x3 window.
    # mode='constant', cval=np.nan ensures the edges of the swath don't wrap around.
    roughness = generic_filter(
        dem_nan, 
        lambda x: np.nanstd(x), 
        size=3, 
        mode='constant', 
        cval=np.nan
    )
    
    # 4. RESTORE NODATA AND EXPORT
    print("Restoring boundaries and exporting matrices...")
    slope_out = np.where(valid_mask, slope_deg, NODATA).astype(np.float32)
    rough_out = np.where(valid_mask, roughness, NODATA).astype(np.float32)
    
    meta.update(dtype=rasterio.float32, nodata=NODATA, compress='lzw')
    
    with rasterio.open(OUT_SLOPE, 'w', **meta) as dst:
        dst.write(slope_out, 1)
        
    with rasterio.open(OUT_ROUGH, 'w', **meta) as dst:
        dst.write(rough_out, 1)
        
    # Validation Printout
    s_valid = slope_out[valid_mask]
    r_valid = rough_out[valid_mask]
    
    print("\n[SUCCESS] Terrain Hazards Generated.")
    print(f"  Slope Stats:     Max = {np.nanmax(s_valid):.2f}°, Mean = {np.nanmean(s_valid):.2f}°")
    print(f"  Roughness Stats: Max = {np.nanmax(r_valid):.2f}m, Mean = {np.nanmean(r_valid):.2f}m")
    print(f"\nSaved: {OUT_SLOPE}")
    print(f"Saved: {OUT_ROUGH}")

# Execute Phase 3
compute_terrain_hazards()

PHASE 3: DERIVING SLOPE AND ROUGHNESS KINEMATICS
Loaded Master DEM. Pixel Resolution: 25.0m
Calculating spatial gradients (Slope)...
Calculating surface roughness (3x3 localized standard deviation)...
  -> (This may take 1-3 minutes on a 19-million pixel grid. Please wait...)
Restoring boundaries and exporting matrices...

[SUCCESS] Terrain Hazards Generated.
  Slope Stats:     Max = 68.04°, Mean = 11.14°
  Roughness Stats: Max = 50.52m, Mean = 4.10m

Saved: C:\DRISHTI_POC\FINAL_DRISHTI_Slope_25m.tif
Saved: C:\DRISHTI_POC\FINAL_DRISHTI_Roughness_25m.tif


### Phase 4: PSR and Doubly Shadowed Crater Detection

##### 1. Scientific Rationale
Permanently Shadowed Regions (PSRs) are areas that have not seen direct sunlight in billions of years whereas a doubly shadowed crater is a localized topographic depression (a nested crater) that exists *entirely inside* a broader PSR. These nested cold traps block not only direct sunlight, but also secondary scattered light bouncing off surrounding illuminated walls, dropping temperatures to < 40 Kelvin.

##### 2. Algorithmic Strategy
1. **The Shadow Mask:** We extract the true PSR bounds using the categorical `PSR_MASK_25m.tif` (where 1.0 = Permanent Shadow).
2. **Topographic Sinks:** We pass a large spatial maximum filter (approx 1 km radius) over the DEM to calculate the "local horizon". Any pixel that is significantly deeper than its local horizon (e.g., > 20 meters deep) is classified as a topographic sink.
3. **Intersection & Labelling:** We intersect the Sinks with the PSR Mask. We then use `scipy.ndimage.label` to isolate contiguous geometries, filter out random noise pixels (keeping only distinct craters > 0.1 km²), and assign each nested crater a unique scientific ID.

In [4]:
import rasterio
import numpy as np
from scipy.ndimage import maximum_filter, label
import os

BASE_DIR = r"C:\DRISHTI_POC"
IN_DEM = os.path.join(BASE_DIR, "LDEM85S_WARPED_25m.tif")
IN_PSR = os.path.join(BASE_DIR, "PSR_MASK_25m.tif")
OUT_DOUBLY_SHADOWED = os.path.join(BASE_DIR, "FINAL_DRISHTI_DoublyShadowed_25m.tif")
NODATA = -9999.0

def detect_doubly_shadowed_craters():
    print("=" * 70)
    print("PHASE 4: EXTRACTING DOUBLY SHADOWED COLD TRAPS")
    print("=" * 70)
    
    with rasterio.open(IN_DEM) as src:
        dem = src.read(1).astype(np.float32)
        meta = src.profile.copy()
        px_size = abs(src.transform.a)
        
    with rasterio.open(IN_PSR) as src:
        psr = src.read(1).astype(np.float32)
        
    valid_mask = (dem != NODATA) & (psr != NODATA)
    dem_nan = np.where(valid_mask, dem, np.nan)
    
    # 1. Isolate the Primary Shadow
    # Assuming PGDA LPSR uses 1 for shadow. We check > 0 to be safe against float conversions.
    is_psr = (psr > 0) & valid_mask
    psr_area_km2 = np.sum(is_psr) * (px_size**2) / 1e6
    print(f"Total PSR Area in Swath: {psr_area_km2:.2f} km²")
    
    # 2. Find Topographic Depressions (Craters)
    print("Scanning for localized topographic sinks (crater detection)...")
    # 41 pixels * 25m = ~1025m window. We are looking for craters up to 1km wide.
    local_horizon = maximum_filter(np.nan_to_num(dem_nan, nan=-99999.0), size=41)
    
    # A pixel is a sink if it is at least 20 meters below its immediate surrounding horizon
    is_depression = (dem_nan < (local_horizon - 20.0)) & valid_mask
    
    # 3. Intersection: Doubly Shadowed
    is_doubly_shadowed = is_psr & is_depression
    
    # 4. Label and Filter Contiguous Craters
    print("Isolating individual nested craters and rejecting thermal noise...")
    labeled_array, num_features = label(is_doubly_shadowed)
    
    final_craters = np.zeros_like(dem, dtype=np.float32)
    valid_crater_count = 0
    
    for crater_id in range(1, num_features + 1):
        crater_mask = (labeled_array == crater_id)
        area_km2 = np.sum(crater_mask) * (px_size**2) / 1e6
        
        # Keep only craters larger than 0.1 sq km to avoid micro-noise
        if area_km2 >= 0.1:
            valid_crater_count += 1
            final_craters[crater_mask] = valid_crater_count # Assign scientific ID
            
            # Print stats for the top 5 largest nested craters
            if valid_crater_count <= 5:
                depth = np.nanmax(dem_nan[crater_mask]) - np.nanmin(dem_nan[crater_mask])
                print(f"  -> Nested Crater {valid_crater_count}: Area = {area_km2:.2f} km², Approx Depth = {depth:.1f}m")
                
    # 5. Export
    final_craters = np.where(valid_mask, final_craters, NODATA)
    meta.update(dtype=rasterio.float32, nodata=NODATA, compress='lzw')
    
    with rasterio.open(OUT_DOUBLY_SHADOWED, 'w', **meta) as dst:
        dst.write(final_craters, 1)
        
    print(f"\n[SUCCESS] Total Significant Doubly Shadowed Craters Found: {valid_crater_count}")
    print(f"Saved: {OUT_DOUBLY_SHADOWED}")

# Execute Phase 4
detect_doubly_shadowed_craters()

PHASE 4: EXTRACTING DOUBLY SHADOWED COLD TRAPS
Total PSR Area in Swath: 3246.25 km²
Scanning for localized topographic sinks (crater detection)...
Isolating individual nested craters and rejecting thermal noise...
  -> Nested Crater 1: Area = 787.15 km², Approx Depth = 2984.0m
  -> Nested Crater 2: Area = 77.98 km², Approx Depth = 760.6m
  -> Nested Crater 3: Area = 21.44 km², Approx Depth = 1342.0m
  -> Nested Crater 4: Area = 0.15 km², Approx Depth = 93.3m
  -> Nested Crater 5: Area = 60.80 km², Approx Depth = 1604.0m

[SUCCESS] Total Significant Doubly Shadowed Craters Found: 437
Saved: C:\DRISHTI_POC\FINAL_DRISHTI_DoublyShadowed_25m.tif


### Phase 5: Multi-Criteria Landing Site Optimization

##### Scientific Rationale
Landing a spacecraft is a trade-off between mission safety and science return. We cannot land directly on the ice because the ice is inside Permanently Shadowed Regions (PSRs); the lander would freeze and its solar panels would fail immediately. 

Therefore, we must find a "sweet spot": a location that receives at least 30% annual solar illumination (for power), has slopes under 10 degrees (for stability), lacks severe boulder roughness, but is mathematically as close as physically possible to the subsurface ice deposits. We also isolate our ROI of the Shoemaker crater from other adjoint basins

##### The Isolated Pixel Problem: Spatial Clustering

A rover should not target isolated, scattered pixels. It needs a continuous, high-density cluster of volatiles.
The Fix: We introduce Phase 4.8. This uses a morphological closing operation to bridge scattered ice chunks within a 100m radius into macro-clusters. We then draw a bounding box around the densest cluster, isolate it, and make it the sole target for Phase 5 and 6.

##### Algorithmic Strategy
We generate a binary admissibility mask where `(Slope <= 10) AND (Roughness <= 8) AND (Illumination >= 0.30)`. We also apply a 1-pixel (25m) safety dilation to the steep slopes to ensure we don't land exactly on the edge of a cliff. 

For all safe pixels, we calculate the Euclidean distance to the nearest ice deposit. The final score uses an exponential decay function ($S = e^{-d / 5000}$), prioritizing safe zones that minimize the rover's traverse distance.

In [11]:
import rasterio
import numpy as np
import os

BASE_DIR = r"C:\DRISHTI_POC"
IN_TRUE_ICE = os.path.join(BASE_DIR, "TRUE_PSR_ICE_MAP_25m.tif")
IN_DSR = os.path.join(BASE_DIR, "FINAL_DRISHTI_DoublyShadowed_25m.tif")
OUT_PRIME_ICE = os.path.join(BASE_DIR, "PRIME_TARGET_ICE_25m.tif")
NODATA = -9999.0

def designate_primary_mission_target():
    print("=" * 70)
    print("PHASE 4.75: DYNAMIC TARGET DESIGNATION (LARGEST BASIN ISOLATION)")
    print("=" * 70)
    
    with rasterio.open(IN_TRUE_ICE) as src:
        ice = src.read(1).astype(np.float32)
        meta = src.profile.copy()
        
    with rasterio.open(IN_DSR) as src:
        dsr = src.read(1).astype(np.float32)
        
    print("Scanning topology for the primary central basin...")
    dsr_ids = np.unique(dsr[(dsr > 0) & (dsr != NODATA)])
    
    max_area, target_basin_id = 0, -1
    for cid in dsr_ids:
        area_pixels = np.sum(dsr == cid)
        if area_pixels > max_area:
            max_area = area_pixels
            target_basin_id = cid
            
    print(f"  -> Largest Basin Found: DSR ID {target_basin_id} (Area: {max_area * 625 / 1e6:.2f} km²)")
    
    prime_ice = np.where((ice > 0) & (dsr == target_basin_id), ice, 0.0)
    prime_ice = np.where(ice == NODATA, NODATA, prime_ice)
    
    target_count = np.sum(prime_ice > 0)
    print(f"\n[SUCCESS] Isolated {target_count:,} Prime Ice pixels inside the primary basin.")
    
    meta.update(dtype=rasterio.float32, nodata=NODATA, compress='lzw')
    with rasterio.open(OUT_PRIME_ICE, 'w', **meta) as dst:
        dst.write(prime_ice, 1)
        
designate_primary_mission_target()

PHASE 4.75: DYNAMIC TARGET DESIGNATION (LARGEST BASIN ISOLATION)
Scanning topology for the primary central basin...
  -> Largest Basin Found: DSR ID 172.0 (Area: 843.32 km²)

[SUCCESS] Isolated 1,021 Prime Ice pixels inside the primary basin.


In [43]:
# ============================================================
# CLUSTER IDENTIFICATION — Run before Phase 5
# Finds the largest connected ice cluster; does NOT modify the
# ice map on disk. TRUE_PSR_ICE_MAP_25m.tif remains untouched.
# ============================================================
import rasterio
import numpy as np
from scipy.ndimage import label
import os

BASE_DIR = r"C:\DRISHTI_POC"
NODATA = -9999.0

IN_TRUE_ICE = os.path.join(BASE_DIR, "TRUE_PSR_ICE_MAP_25m.tif")
IN_ROI = os.path.join(BASE_DIR, "SHOEMAKER_ROI_MASK.tif")
OUT_CLUSTER_LABELS = os.path.join(BASE_DIR, "ICE_CLUSTER_LABELS.tif")

def identify_primary_ice_cluster():
    print("=" * 70)
    print("ICE CLUSTER ANALYSIS — Identifying primary science target")
    print("=" * 70)

    with rasterio.open(IN_TRUE_ICE) as src:
        ice = src.read(1).astype(np.float32)
        meta = src.profile.copy()
        px = abs(src.transform.a)
        transform = src.transform

    with rasterio.open(IN_ROI) as src:
        roi = src.read(1).astype(np.float32)

    ice_binary = (ice > 0) & (ice != NODATA)
    print(f"Total authentic ice pixels (full swath): {np.sum(ice_binary):,}")

    # 8-connectivity — diagonal neighbors count as connected
    structure = np.ones((3, 3))
    labeled, n_clusters = label(ice_binary, structure=structure)
    print(f"Distinct connected ice clusters found: {n_clusters}")

    cluster_stats = []
    for cid in range(1, n_clusters + 1):
        mask = labeled == cid
        area_px = np.sum(mask)
        if area_px < 2:
            continue  # ignore single-pixel noise
        rows, cols = np.where(mask)
        centroid_r, centroid_c = rows.mean(), cols.mean()
        cx, cy = rasterio.transform.xy(transform, centroid_r, centroid_c)
        inside_roi = roi[int(round(centroid_r)), int(round(centroid_c))] > 0
        cluster_stats.append({
            "id": cid, "area_px": area_px, "area_m2": area_px * px * px,
            "centroid_row": centroid_r, "centroid_col": centroid_c,
            "centroid_x": cx, "centroid_y": cy, "inside_shoemaker_roi": inside_roi
        })

    cluster_stats.sort(key=lambda d: -d["area_px"])

    print(f"\nTop 10 clusters by area:")
    print(f"{'Rank':<5}{'ID':<6}{'Pixels':<8}{'Area(m²)':<12}{'In ROI?':<10}")
    for i, c in enumerate(cluster_stats[:10]):
        print(f"{i+1:<5}{c['id']:<6}{c['area_px']:<8}{c['area_m2']:<12.0f}"
              f"{'YES' if c['inside_shoemaker_roi'] else 'no':<10}")

    # Primary target = largest cluster that is ALSO inside the Shoemaker ROI
    roi_clusters = [c for c in cluster_stats if c["inside_shoemaker_roi"]]
    if not roi_clusters:
        print("\n[CRITICAL] No ice cluster centroid falls inside the Shoemaker ROI.")
        print("This would mean Shoemaker's own ice is fragmented below the ROI")
        print("boundary, or the ROI itself needs re-examination — not a targeting fix.")
        return None, labeled, meta

    primary = roi_clusters[0]
    print(f"\n[SELECTED] Primary science cluster: ID {primary['id']}")
    print(f"  Area: {primary['area_px']} pixels ({primary['area_m2']/1e6:.3f} km²)")
    print(f"  Centroid: row={primary['centroid_row']:.0f}, "
          f"col={primary['centroid_col']:.0f}")
    print(f"  Note: this is the LARGEST connected ice cluster whose centroid")
    print(f"  falls inside the Shoemaker ROI — selected by area ranking,")
    print(f"  not by manual override.")

    # Save the label map purely for QGIS inspection — full map preserved
    meta.update(dtype='float32', nodata=NODATA, compress='lzw')
    label_out = np.where(ice_binary, labeled.astype(np.float32), NODATA)
    with rasterio.open(OUT_CLUSTER_LABELS, 'w', **meta) as dst:
        dst.write(label_out, 1)

    return primary["id"], labeled, meta

PRIMARY_CLUSTER_ID, CLUSTER_LABELS, _ = identify_primary_ice_cluster()

ICE CLUSTER ANALYSIS — Identifying primary science target
Total authentic ice pixels (full swath): 2,038
Distinct connected ice clusters found: 196

Top 10 clusters by area:
Rank ID    Pixels  Area(m²)    In ROI?   
1    149   112     70000       YES       
2    91    79      49375       YES       
3    14    52      32500       no        
4    152   50      31250       YES       
5    84    47      29375       YES       
6    190   47      29375       no        
7    77    39      24375       YES       
8    16    34      21250       no        
9    103   33      20625       no        
10   183   28      17500       no        

[SELECTED] Primary science cluster: ID 149
  Area: 112 pixels (0.070 km²)
  Centroid: row=1939, col=3172
  Note: this is the LARGEST connected ice cluster whose centroid
  falls inside the Shoemaker ROI — selected by area ranking,
  not by manual override.


In [30]:
LUNAR_SYNODIC_DAY_HOURS = 29.5 * 24   # ~708 hours, fixed orbital mechanics
VIKRAM_OPERATIONAL_DESIGN_HOURS = 14 * 24  # ISRO's stated one-lunar-day design life

def estimate_operational_window(avgvisib_value):
    """
    Converts the average illumination fraction into an expected number
    of sunlit hours per synodic period — an honest, derivable physical
    quantity, not a fabricated schedule.
    
    This does NOT claim to know exact eclipse timing. It states the
    statistical expectation, citing Mazarico et al. (2011) methodology.
    """
    expected_sunlit_hours = avgvisib_value * LUNAR_SYNODIC_DAY_HOURS
    expected_dark_hours = LUNAR_SYNODIC_DAY_HOURS - expected_sunlit_hours

    meets_vikram_design = expected_sunlit_hours >= VIKRAM_OPERATIONAL_DESIGN_HOURS

    return {
        "expected_sunlit_hours_per_synodic_period": expected_sunlit_hours,
        "expected_dark_hours_per_synodic_period": expected_dark_hours,
        "meets_chandrayaan3_design_precedent": meets_vikram_design
    }

In [31]:
PRAGYAN_SPEED_MPS = 0.01  # 1 cm/s, published ISRO Chandrayaan-3 rover spec

def evaluate_traverse_energy_budget(path_coords_df, avgvisib_along_path,
                                     operational_window):
    """
    path_coords_df: output of your A* Rover_Path_Coordinates.csv
    avgvisib_along_path: AVGVISIB sampled at each path pixel
    """
    total_distance_m = 0.0
    for i in range(1, len(path_coords_df)):
        dx = path_coords_df.iloc[i]['X_Easting'] - path_coords_df.iloc[i-1]['X_Easting']
        dy = path_coords_df.iloc[i]['Y_Northing'] - path_coords_df.iloc[i-1]['Y_Northing']
        total_distance_m += np.sqrt(dx**2 + dy**2)

    traverse_time_s = total_distance_m / PRAGYAN_SPEED_MPS
    traverse_time_hours = traverse_time_s / 3600.0

    sunlit_hours_available = operational_window["expected_sunlit_hours_per_synodic_period"]

    print(f"Total traverse distance: {total_distance_m:.1f} m")
    print(f"Traverse time at Pragyan-class speed: {traverse_time_hours:.1f} hours")
    print(f"Sunlit hours available (synodic estimate): {sunlit_hours_available:.1f} hours")

    if traverse_time_hours > sunlit_hours_available:
        print("⚠ Traverse duration EXCEEDS expected illuminated window.")
        print("  Mission would require multi-day operation with recharge holds,")
        print("  consistent with how VIPER's mission plan budgets PSR excursions")
        print("  against remaining battery reserve, not continuous traversal.")
    else:
        print("✅ Traverse fits within single illuminated operational period.")

    return traverse_time_hours

In [23]:
# ============================================================
#  — Keep Gaussian heatmap ONLY as a visualization
# layer, never as a landing-site input
# ============================================================
import rasterio
import numpy as np
from scipy.ndimage import gaussian_filter
import os

BASE_DIR = r"C:\DRISHTI_POC"
IN_TRUE_ICE = os.path.join(BASE_DIR, "TRUE_PSR_ICE_MAP_25m.tif")
OUT_DENSITY_TIF = os.path.join(BASE_DIR, "ICE_DENSITY_HEATMAP_25m.tif")
NODATA = -9999.0

def generate_density_visualization_only():
    """
    PURELY COSMETIC. Used for presentation heatmaps only.
    This output must NEVER be fed into Phase 5 or 6 as a target.
    """
    with rasterio.open(IN_TRUE_ICE) as src:
        ice = src.read(1).astype(np.float32)
        meta = src.profile.copy()

    ice_binary = np.where((ice > 0) & (ice != NODATA), 1.0, 0.0)
    density = gaussian_filter(ice_binary, sigma=40)
    density = np.where(ice == NODATA, NODATA, density)

    meta.update(dtype=rasterio.float32, nodata=NODATA, compress='lzw')
    with rasterio.open(OUT_DENSITY_TIF, 'w', **meta) as dst:
        dst.write(density, 1)
    print("[VISUALIZATION ONLY] Density heatmap saved for presentation slides.")

generate_density_visualization_only()

[VISUALIZATION ONLY] Density heatmap saved for presentation slides.


### Architecture - A: Navigate without illumination constraint(PSRs are hardly illuminated by the Sun)- Use RTGs for charge

In [59]:
import rasterio
import numpy as np
from scipy.ndimage import distance_transform_edt, binary_dilation
from skimage.graph import MCP_Geometric
import os

BASE_DIR = r"C:\DRISHTI_POC"
NODATA = -9999.0

IN_SLOPE = os.path.join(BASE_DIR, "FINAL_DRISHTI_Slope_25m.tif")
IN_ROUGH = os.path.join(BASE_DIR, "FINAL_DRISHTI_Roughness_25m.tif")
IN_ILLUM = os.path.join(BASE_DIR, "AVGVISIB_WARPED_25m.tif")
IN_LABELS = os.path.join(BASE_DIR, "ICE_CLUSTER_LABELS.tif")
IN_DEM   = os.path.join(BASE_DIR, "LDEM85S_WARPED_25m.tif")
IN_PSR   = os.path.join(BASE_DIR, "PSR_MASK_25m.tif")

OUT_LANDING = os.path.join(BASE_DIR, "FINAL_DRISHTI_LandingScore_25m.tif")

# Target the Primary Cluster
PRIMARY_CLUSTER_ID = 149 

def compute_kinematic_landing_site():
    print("=" * 70)
    print("PHASE 5: REVERSE KINEMATIC OPTIMIZATION (FULL RTG ARCHITECTURE)")
    print("=" * 70)
    
    def load_raster(path):
        with rasterio.open(path) as src:
            arr = src.read(1).astype(np.float32)
            meta = src.profile.copy()
            if src.nodata is not None: arr[arr == src.nodata] = np.nan
            return arr, meta

    print("Loading Artemis Terrain Constraints...")
    slope, meta = load_raster(IN_SLOPE)
    rough, _ = load_raster(IN_ROUGH)
    illum, _ = load_raster(IN_ILLUM)
    labels, _ = load_raster(IN_LABELS)
    psr, _ = load_raster(IN_PSR)
    
    footprint = ~np.isnan(slope) & ~np.isnan(rough) & ~np.isnan(illum)
    
    # 1. BUILD THE BATTERY-DASH FRICTION SURFACE (Mechanical Only)
    print("Computing mechanical friction surface (Illumination penalty removed)...")
    W_SLOPE = 10.0
    W_ROUGH = 5.0
    
    friction = np.full(slope.shape, np.inf, dtype=np.float32)
    
    slope_pen = W_SLOPE * np.exp((slope / 15.0)**2)
    rough_pen = W_ROUGH * (rough / 5.0)**2
    
    valid_terrain = (slope <= 15.0) & (rough <= 5.0) & footprint
    friction[valid_terrain] = 25.0 * (1.0 + slope_pen[valid_terrain] + rough_pen[valid_terrain])
    
    # 2. EXECUTE REVERSE COST-DISTANCE FLOOD
    print(f"Initiating Cost-Distance Flood from Primary Target (Cluster {PRIMARY_CLUSTER_ID})...")
    target_mask = (labels == PRIMARY_CLUSTER_ID)
    start_indices = np.column_stack(np.where(target_mask))
    
    if len(start_indices) == 0:
        print("[CRITICAL] Primary target cluster not found!")
        return
        
    mcp = MCP_Geometric(friction)
    cumulative_costs, _ = mcp.find_costs(start_indices)
    
    # 3. APPLY HARD LANDING GATES (RTG Rules - No Power Constraint)
    print("Filtering accessible sites against RTG Landing Constraints...")
    safe_slope = (slope <= 10.0)
    safe_rough = (rough <= 8.0)
    
    # PSR Avoidance: Must land outside strict pitch-black PSR for optical descent nav
    psr_hazard = (psr > 0)
    psr_buffer = binary_dilation(psr_hazard, structure=np.ones((3,3)), iterations=1)
    safe_from_shadow = ~psr_buffer
    
    # Geographic Plume Standoff (must be 500m away physically)
    dist_px = distance_transform_edt(~target_mask)
    dist_m = dist_px * 25.0
    safe_plume_dist = (dist_m >= 500.0)
    
    # MUST BE REACHABLE
    reachable = (cumulative_costs < np.inf)

    # REMOVED `safe_power` gate to allow landing in dim/twilight regions
    admissible = safe_slope & safe_rough & safe_from_shadow & safe_plume_dist & reachable & footprint
    safe_pixels = np.sum(admissible)
    
    if safe_pixels == 0:
        print("\n[MISSION ABORT] Zero accessible landing sites exist.")
        return
        
    print(f"Validated Accessible RTG Sites: {safe_pixels:,} pixels")
    
    # Optimal site minimizes mechanical energy drain on the rover
    search_matrix = np.where(admissible, cumulative_costs, np.inf)
    best_idx = np.unravel_index(np.argmin(search_matrix), search_matrix.shape)
    
    print("\n" + "=" * 50)
    print("🏆 RTG LANDING SITE SECURED")
    print("=" * 50)
    print(f"  Matrix Coordinates: (Row {best_idx[0]}, Col {best_idx[1]})")
    print(f"  Local Slope:        {slope[best_idx]:.2f}°")
    print(f"  Geographic Dist:    {dist_m[best_idx]:.1f} meters")
    print(f"  Kinematic Cost:     {cumulative_costs[best_idx]:.1f} mechanical units")
    
    max_valid_cost = np.nanmax(np.where(admissible, cumulative_costs, np.nan))
    landing_score = np.where(admissible, np.exp(-cumulative_costs / max_valid_cost), 0.0)
    
    meta.update(dtype=rasterio.float32, nodata=NODATA, compress='lzw')
    with rasterio.open(OUT_LANDING, 'w', **meta) as dst:
        dst.write(landing_score.astype(np.float32), 1)

compute_kinematic_landing_site()

PHASE 5: REVERSE KINEMATIC OPTIMIZATION (FULL RTG ARCHITECTURE)
Loading Artemis Terrain Constraints...
Computing mechanical friction surface (Illumination penalty removed)...
Initiating Cost-Distance Flood from Primary Target (Cluster 149)...
Filtering accessible sites against RTG Landing Constraints...
Validated Accessible RTG Sites: 4,668 pixels

🏆 RTG LANDING SITE SECURED
  Matrix Coordinates: (Row 2650, Col 2872)
  Local Slope:        9.13°
  Geographic Dist:    19010.4 meters
  Kinematic Cost:     250111.0 mechanical units


### Architecture - B: Consider illumination map over terrain for solar charging substation and then travel miles to the target site

In [51]:
import rasterio
import numpy as np
import pyproj

BASE_DIR = r"C:\DRISHTI_POC"
REF_TIF = f"{BASE_DIR}/WARPED_CPR.tif"

with rasterio.open(REF_TIF) as src:
    crs = src.crs
    transform = src.transform

moon_geog = pyproj.CRS.from_proj4("+proj=longlat +R=1737400 +no_defs")
transformer = pyproj.Transformer.from_crs(moon_geog, crs, always_xy=True)
cx, cy = transformer.transform(44.9, -87.7)

# Top-right rover marker pixel from your screenshot — read it off the QGIS
# coordinate readout when you hover the marker, plug in row/col here:
marker_row, marker_col = 6, 5717   # EXAMPLE — replace with actual hover readout
mx, my = rasterio.transform.xy(transform, marker_row, marker_col)

dist_km = np.sqrt((mx - cx)**2 + (my - cy)**2) / 1000.0
print(f"Shoemaker center: ({cx:.0f}, {cy:.0f})")
print(f"Marker location:  ({mx:.0f}, {my:.0f})")
print(f"Distance from Shoemaker center: {dist_km:.1f} km")
print(f"With MARGIN_KM=4.0, ROI radius = {25.5+4.0:.1f} km — "
      f"marker {'INSIDE' if dist_km <= 29.5 else 'OUTSIDE'} the new ROI")

Shoemaker center: (49237, 49409)
Marker location:  (115464, 89891)
Distance from Shoemaker center: 77.6 km
With MARGIN_KM=4.0, ROI radius = 29.5 km — marker OUTSIDE the new ROI


### Phase 6: Dual-Tier Autonomous Rover Pathfinding 

##### 1. Scientific Rationale: The Static vs. Dynamic Navigation Problem
Our Phase 5 orbital map operates at a 25-meter resolution. While mathematically robust, a 25m pixel can easily hide a 2-meter boulder. If we force the rover to blindly follow a static orbital path, it will crash into unseen micro-hazards. 

Therefore, DRISHTI implements a space-grade **Dual-Tier Navigation Architecture**, mimicking the logic of the Pragyan and Perseverance rovers:
1. **Global Strategic Planner (Ground Control):** Utilizes the **A* (A-Star) Algorithm** on the 25m DFSAR/LOLA cost map to generate a long-distance, mathematically optimal sequence of waypoints.
2. **Local Tactical Planner (Onboard Rover):** The rover's internal chassis computer utilizes the **D* Lite (Dynamic A*) Algorithm**. As the rover drives, its onboard NavCams sweep the terrain. If an unmapped micro-obstacle is detected, D* Lite dynamically recalculates the local path in $O(k \log n)$ time, maneuvering around the hazard without needing to re-solve the entire orbital map.

##### 2. Algorithmic Execution
In this cell, we execute the **Ground Control Global Planner (A*)**. 
We ingest the Topographic Slope, Roughness, and Illumination matrices to build a non-linear cost friction surface. Impassable slopes ($>15^\circ$) are classified as infinite cost. The algorithm dynamically crops a bounding box between the landing site and the deep PSR ice target, computing the lowest-energy macroscopic path to serve as the baseline for the rover's onboard D* Lite systems.

In [60]:
import rasterio
import numpy as np
import pandas as pd
from skimage.graph import MCP_Geometric
import os

BASE_DIR = r"C:\DRISHTI_POC"
NODATA = -9999.0

IN_SLOPE = os.path.join(BASE_DIR, "FINAL_DRISHTI_Slope_25m.tif")
IN_ROUGH = os.path.join(BASE_DIR, "FINAL_DRISHTI_Roughness_25m.tif")
IN_LABELS = os.path.join(BASE_DIR, "ICE_CLUSTER_LABELS.tif")
IN_DEM   = os.path.join(BASE_DIR, "LDEM85S_WARPED_25m.tif")
IN_LANDING = os.path.join(BASE_DIR, "FINAL_DRISHTI_LandingScore_25m.tif")

OUT_PATH_CSV = os.path.join(BASE_DIR, "Rover_Path_Coordinates.csv")
PRIMARY_CLUSTER_ID = 149

def extract_proven_trajectory():
    print("=" * 70)
    print("PHASE 6: ARTEMIS-GRADE SURFACE MOBILITY (BATTERY DASH)")
    print("=" * 70)
    
    def load(path):
        with rasterio.open(path) as src:
            return src.read(1).astype(np.float32), src.transform, src.profile
            
    slope, transform, meta = load(IN_SLOPE)
    rough, _, _ = load(IN_ROUGH)
    labels, _, _ = load(IN_LABELS)
    dem, _, _ = load(IN_DEM)
    landing, _, _ = load(IN_LANDING)
    
    # 1. Locate the Substation Lander
    start_idx = np.unravel_index(np.argmax(landing), landing.shape)
    
    # 2. Rebuild the Mechanical Friction Surface (Must identically match Phase 5)
    W_SLOPE = 10.0
    W_ROUGH = 5.0
    
    friction = np.full(slope.shape, np.inf, dtype=np.float32)
    valid = (slope <= 15.0) & (rough <= 5.0) & (slope != NODATA)
    
    slope_pen = W_SLOPE * np.exp((slope / 15.0)**2)
    rough_pen = W_ROUGH * (rough / 5.0)**2
    
    friction[valid] = 25.0 * (1.0 + slope_pen[valid] + rough_pen[valid])
    
    # 3. Extract the Traceback
    print("Extracting verified trajectory from mechanical kinematic field...")
    target_mask = (labels == PRIMARY_CLUSTER_ID)
    ice_indices = np.column_stack(np.where(target_mask))
    
    mcp = MCP_Geometric(friction)
    mcp.find_costs(ice_indices)
    
    # Traceback from the Lander DOWN to the Ice
    path_indices = mcp.traceback(start_idx)
    
    # 4. Export the Data
    path_coords = []
    for p in path_indices:
        r, c = p[0], p[1]
        x, y = transform * (c, r)
        path_coords.append({
            "Row": r, "Col": c, 
            "X_Easting": x, "Y_Northing": y, 
            "Elevation": dem[r, c], "Slope": slope[r, c]
        })

    pd.DataFrame(path_coords).to_csv(OUT_PATH_CSV, index=False)
    
    physical_distance = (len(path_indices) - 1) * 25.0
    print(f"\n[SUCCESS] Trajectory Locked and Exported.")
    print(f"  Substation Location: {start_idx}")
    print(f"  Terminal Ice Node:   {path_indices[-1]}")
    print(f"  Total Traverse:      {physical_distance:.1f} meters")
    print(f"  Maximum Slope Hit:   {max([p['Slope'] for p in path_coords]):.2f}°")

extract_proven_trajectory()

PHASE 6: ARTEMIS-GRADE SURFACE MOBILITY (BATTERY DASH)
Extracting verified trajectory from mechanical kinematic field...

[SUCCESS] Trajectory Locked and Exported.
  Substation Location: (np.int64(2650), np.int64(2872))
  Terminal Ice Node:   (2650, 2872)
  Total Traverse:      17900.0 meters
  Maximum Slope Hit:   13.51°


In [52]:
# ============================================================
# COST SURFACE EXPORT — run once, alongside Phase 6
# Produces the explainability layer you're missing
# ============================================================
import rasterio
import numpy as np
import os

BASE_DIR = r"C:\DRISHTI_POC"
NODATA = -9999.0

IN_SLOPE = os.path.join(BASE_DIR, "FINAL_DRISHTI_Slope_25m.tif")
IN_ROUGH = os.path.join(BASE_DIR, "FINAL_DRISHTI_Roughness_25m.tif")
IN_ILLUM = os.path.join(BASE_DIR, "AVGVISIB_WARPED_25m.tif")
IN_ROI   = os.path.join(BASE_DIR, "SHOEMAKER_ROI_MASK.tif")

OUT_COST_SLOPE = os.path.join(BASE_DIR, "COST_Slope_Component.tif")
OUT_COST_ROUGH = os.path.join(BASE_DIR, "COST_Roughness_Component.tif")
OUT_COST_ILLUM = os.path.join(BASE_DIR, "COST_Illumination_Component.tif")
OUT_COST_TOTAL = os.path.join(BASE_DIR, "COST_TOTAL_Surface.tif")

THETA_CRIT, ROUGH_CRIT = 10.0, 5.0
W_SLOPE, W_ROUGH, W_ILLUM = 10.0, 5.0, 8.0
HARD_SLOPE_LIMIT = 15.0
ILLUM_MIN_ABS = 0.30

def build_cost_surface():
    def load(p):
        with rasterio.open(p) as s:
            return s.read(1).astype(np.float32), s.profile.copy()

    slope, meta = load(IN_SLOPE)
    rough, _ = load(IN_ROUGH)
    illum, _ = load(IN_ILLUM)
    roi, _ = load(IN_ROI)

    valid = (slope != NODATA) & (rough != NODATA) & (illum != NODATA) & (roi > 0)

    slip_penalty = np.where(valid, W_SLOPE * np.exp((slope/THETA_CRIT)**2) - W_SLOPE, np.nan)
    rough_penalty = np.where(valid, W_ROUGH * (rough/ROUGH_CRIT)**2, np.nan)
    illum_penalty = np.where(
        valid, W_ILLUM * np.maximum(0.0, (ILLUM_MIN_ABS - illum)) / ILLUM_MIN_ABS, np.nan
    )

    # Mark impassable cells explicitly so the visualization shows hard walls
    impassable = (slope > HARD_SLOPE_LIMIT) & valid
    total_cost = np.where(valid, 1.0 + slip_penalty + rough_penalty + illum_penalty, np.nan)
    total_cost = np.where(impassable, np.nan, total_cost)  # render as transparent voids

    def write(path, arr, meta):
        out = np.where(np.isnan(arr), NODATA, arr).astype(np.float32)
        m = meta.copy()
        m.update(dtype='float32', nodata=NODATA, compress='lzw')
        with rasterio.open(path, 'w', **m) as dst:
            dst.write(out, 1)

    write(OUT_COST_SLOPE, slip_penalty, meta)
    write(OUT_COST_ROUGH, rough_penalty, meta)
    write(OUT_COST_ILLUM, illum_penalty, meta)
    write(OUT_COST_TOTAL, total_cost, meta)

    print("Cost surface components written:")
    print(f"  Slope cost range:  {np.nanmin(slip_penalty):.2f} – {np.nanmax(slip_penalty):.2f}")
    print(f"  Rough cost range:  {np.nanmin(rough_penalty):.2f} – {np.nanmax(rough_penalty):.2f}")
    print(f"  Illum cost range:  {np.nanmin(illum_penalty):.2f} – {np.nanmax(illum_penalty):.2f}")
    print(f"  Total cost range:  {np.nanmin(total_cost):.2f} – {np.nanmax(total_cost):.2f}")
    print(f"  Impassable cells:  {np.sum(impassable):,}")

build_cost_surface()

Cost surface components written:
  Slope cost range:  0.00 – 19197031612416.00
  Rough cost range:  0.00 – 106.80
  Illum cost range:  0.00 – 8.00
  Total cost range:  1.00 – 109.77
  Impassable cells:  1,180,927


## Phase 4.5: Thermal Pruning of Radar Ambiguity (Relate Back2 Science)

### 1. Scientific Rationale
In Phase 1, we utilized the T3 Coherency Matrix and Yamaguchi Decomposition alongside CPR and DOP to isolate regions of high volume and double-bounce scattering—the primary structural signatures of subsurface volatiles. 

However, microwave radar suffers from dielectric ambiguity: wavelength-scale surface roughness (buried rocky ejecta) generates identical scattering profiles to trapped water-ice. 

### 2. Algorithmic Strategy (Breaking the Ambiguity)
To isolate true volatiles, we fuse the structural radar data with thermal bounding. We mathematically intersect the `DRISHTI_FINAL_ICE_MAP` with the Phase 4 `PSR_MASK`. 
Thermodynamics dictates that any high-scattering anomaly located outside a Permanently Shadowed Region (PSR) is subjected to extreme thermal cycling and sublimation; thus, it is a false positive (a boulder field). Anomalies strictly confined within the PSR bounds are thermally stable and are reclassified as High-Confidence Volatile Targets.

In [4]:
import rasterio
import numpy as np
import os

BASE_DIR = r"C:\DRISHTI_POC"
IN_RAW_ICE = os.path.join(BASE_DIR, "DRISHTI_FINAL_ICE_MAP.tif")
IN_PSR = os.path.join(BASE_DIR, "PSR_MASK_25m.tif")
OUT_TRUE_ICE = os.path.join(BASE_DIR, "PRIME_TARGET_ICE_25m.tif")
NODATA = -9999.0

def prune_radar_false_positives():
    print("=" * 70)
    print("PHASE 4.5: THERMAL PRUNING OF YAMAGUCHI ICE MAP (CORRECTED)")
    print("=" * 70)
    
    with rasterio.open(IN_RAW_ICE) as src:
        raw_ice = src.read(1).astype(np.float32)
        meta = src.profile.copy()
        
    with rasterio.open(IN_PSR) as src:
        psr_mask = src.read(1).astype(np.float32)
        
    # Count original Yamaguchi anomalies
    orig_count = np.sum((raw_ice > 0) & (raw_ice != NODATA))
    
    # THE FIX: Keep ice ONLY if the PSR mask is > 0 (Handling the 20000.0 value)
    true_ice = np.where((raw_ice > 0) & (psr_mask > 0), raw_ice, 0.0)
    
    # Restore strict NODATA bounds
    true_ice = np.where(raw_ice == NODATA, NODATA, true_ice)
    
    pruned_count = np.sum(true_ice > 0)
    false_positives = orig_count - pruned_count
    
    print(f"Original Radar Anomalies: {orig_count:,} pixels")
    print(f"Thermally Validated Ice:  {pruned_count:,} pixels")
    print(f"False Positives Erased:   {false_positives:,} pixels (Discarded as rocky ejecta)")
    
    meta.update(dtype=rasterio.float32, nodata=NODATA, compress='lzw')
    with rasterio.open(OUT_TRUE_ICE, 'w', **meta) as dst:
        dst.write(true_ice, 1)
        
    print(f"\n[SUCCESS] Pruned Ice Map saved to: {OUT_TRUE_ICE}")

# Execute
prune_radar_false_positives()

PHASE 4.5: THERMAL PRUNING OF YAMAGUCHI ICE MAP (CORRECTED)
Original Radar Anomalies: 5,385 pixels
Thermally Validated Ice:  2,038 pixels
False Positives Erased:   3,347 pixels (Discarded as rocky ejecta)

[SUCCESS] Pruned Ice Map saved to: C:\DRISHTI_POC\PRIME_TARGET_ICE_25m.tif


### Phase 7: 3D Interactive Mission Visualization

##### Overview
This cell utilizes Plotly and WebGL to render a 3D topographical mesh of the target region. It extracts the raw elevation data, plots the landing site, traces the kinematic rover traverse, and highlights the target Doubly Shadowed crater. This allows mission planners to visually verify the obstacle-avoidance logic calculated by the pathfinding algorithm.

In [28]:
import pandas as pd
import rasterio
import numpy as np
import plotly.graph_objects as go
import os

BASE_DIR = r"C:\DRISHTI_POC"
NODATA = -9999.0

print("Loading Authentic DRISHTI Data Matrices...")

# 1. Load the Path and Arrays
df_path = pd.read_csv(os.path.join(BASE_DIR, "Rover_Path_Coordinates.csv"))

def load_array(filename):
    with rasterio.open(os.path.join(BASE_DIR, filename)) as src:
        return src.read(1).astype(np.float32)

dem   = load_array("LDEM85S_WARPED_25m.tif")
slope = load_array("FINAL_DRISHTI_Slope_25m.tif")
ice   = load_array("TRUE_PSR_ICE_MAP_25m.tif")

# 2. Dynamic Cropping (Bounding Box around the path + 80 pixel buffer for context)
min_row, max_row = int(df_path['Row'].min()) - 80, int(df_path['Row'].max()) + 80
min_col, max_col = int(df_path['Col'].min()) - 80, int(df_path['Col'].max()) + 80

min_row, max_row = max(0, min_row), min(dem.shape[0], max_row)
min_col, max_col = max(0, min_col), min(dem.shape[1], max_col)

dem_c   = dem[min_row:max_row, min_col:max_col]
slope_c = slope[min_row:max_row, min_col:max_col]
ice_c   = ice[min_row:max_row, min_col:max_col]

# Mask NODATA
dem_c   = np.where(dem_c == NODATA, np.nan, dem_c)
slope_c = np.where(slope_c == NODATA, np.nan, slope_c)

# 3. Generate 3D Grid Coordinates
x = np.arange(0, dem_c.shape[1] * 25, 25)
y = np.arange(0, dem_c.shape[0] * 25, 25)
X, Y = np.meshgrid(x, y)

# Path relative coordinates
path_x = (df_path['Col'] - min_col) * 25
path_y = (df_path['Row'] - min_row) * 25
path_z = df_path['Elevation'] + 1.0 # Float 1 meter above surface

# Authentic Ice pixels in this zone
ice_y_idx, ice_x_idx = np.where(ice_c > 0)
ice_x = ice_x_idx * 25
ice_y = ice_y_idx * 25
ice_z = dem_c[ice_y_idx, ice_x_idx] + 0.5 

print("Compiling WebGL Mesh with Lunar Illumination Physics...")

fig = go.Figure()

# --- TRACE 0: The Base Topography with Lunar Lighting ---
fig.add_trace(go.Surface(
    z=dem_c,
    x=X, y=Y,
    surfacecolor=dem_c,
    colorscale='Greys', # Lunar realism
    cmin=np.nanmin(dem_c), cmax=np.nanmax(dem_c),
    colorbar=dict(title="Elevation (m)", x=-0.1),
    # THIS IS THE FIX: Analytical lighting to mimic QGIS hillshade
    lighting=dict(ambient=0.3, diffuse=0.8, roughness=0.8, specular=0.1, fresnel=0.1),
    lightposition=dict(x=1000, y=1000, z=500) # Simulates a grazing sun angle
))

# --- TRACE 1: Authentic Ice Deposits ---
fig.add_trace(go.Scatter3d(
    x=ice_x, y=ice_y, z=ice_z,
    mode='markers',
    marker=dict(size=4, color='#00FFFF', opacity=0.8, symbol='circle'),
    name='Verified Volatiles'
))

# --- TRACE 2: Kinematic Rover Path ---
fig.add_trace(go.Scatter3d(
    x=path_x, y=path_y, z=path_z,
    mode='lines',
    line=dict(color='#39FF14', width=6), # Neon green for contrast
    name='A* Kinematic Traverse'
))

# --- TRACE 3: Lander ---
fig.add_trace(go.Scatter3d(
    x=[path_x.iloc[0]], y=[path_y.iloc[0]], z=[path_z.iloc[0]],
    mode='markers',
    marker=dict(size=8, color='yellow', symbol='diamond'),
    name='Vikram Lander'
))

# --- UI DROPDOWN: Multi-Layer Matrix Toggling ---
fig.update_layout(
    updatemenus=[
        dict(
            buttons=list([
                dict(
                    args=[{"surfacecolor": [dem_c], "colorscale": "Greys", "cmin": np.nanmin(dem_c), "cmax": np.nanmax(dem_c)}],
                    label="Layer: Lunar Hillshade (DEM)",
                    method="restyle"
                ),
                dict(
                    args=[{"surfacecolor": [slope_c], "colorscale": "Inferno", "cmin": 0, "cmax": 15}],
                    label="Layer: Slope Hazard Matrix",
                    method="restyle"
                )
            ]),
            direction="down",
            pad={"r": 10, "t": 10},
            showactive=True,
            x=0.05,
            xanchor="left",
            y=1.1,
            yanchor="top"
        )
    ]
)

# Configure Camera and Layout
fig.update_layout(
    title="DRISHTI: Authentic South Pole Rover Simulation",
    scene=dict(
        xaxis_title='Easting (m)',
        yaxis_title='Northing (m)',
        zaxis_title='Elevation (m)',
        aspectmode='data', # Locks true physical scaling (no stretching)
        camera=dict(eye=dict(x=1.2, y=1.2, z=0.6))
    ),
    width=1000,
    height=800,
    margin=dict(l=0, r=0, b=0, t=50),
    template="plotly_dark"
)

fig.show(renderer='browser')

Loading Authentic DRISHTI Data Matrices...
Compiling WebGL Mesh with Lunar Illumination Physics...
